<a href="https://colab.research.google.com/github/filipsajtlava/dspracticum2025-tismaci/blob/master/homeworks/hw6/Llama3.2_3B_Conversational.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


In [ ]:
!git clone https://github.com/filipsajtlava/dspracticum2025-tismaci
%cd dspracticum2025-tismaci/

In [ ]:
file_path = "homeworks/hw4/textovy_dataset.txt"
with open(file_path, 'r') as f:
    textovy_dataset_content = f.read()

print("Content loaded successfully. First 200 characters:")
print(textovy_dataset_content[:200])

In [ ]:
# New Installation Cell: Ensuring unsloth and trl are correctly installed.
# Removed %%capture temporarily to show installation output for debugging.

import os, re

print("Starting installation of Unsloth and dependencies...")

if "COLAB_" not in "".join(os.environ.keys()):
    print("Installing unsloth for non-Colab environment...")
    !pip install unsloth
else:
    print("Installing unsloth and dependencies for Colab environment...")
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers_version = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")

    print(f"Installing core libraries including xformers ({xformers_version})...")
    !pip install bitsandbytes accelerate peft triton cut_cross_entropy unsloth_zoo {xformers_version}

    print("Installing trl...")
    !pip install trl

    print("Installing unsloth...")
    !pip install unsloth

    print("Installing other data processing dependencies...")
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer

print("Installation process concluded.")

In [ ]:
# New Cell for Model Loading (from original Znh96WvcstqF)

from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-bnb-4bit", # Changed to base model
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

In [ ]:
# New Cell for LoRA Adapters (from original 6bZsfBuZDeCL)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
# New Data Preparation Cell for Continued Pretraining (Pure Text)

from unsloth.chat_templates import get_chat_template
from datasets import Dataset

# Ensure the tokenizer is loaded with the chat template for potential inference later,
# but this template is NOT applied to the training data itself for continued pretraining.
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1", # Use the appropriate chat template for your model
)

# Create a dataset where the 'text' field contains your entire pure text content.
# We assume `textovy_dataset_content` is already loaded into the environment.
dataset_raw = [{
    "text": textovy_dataset_content
}]
dataset = Dataset.from_list(dataset_raw)

print("Dataset created for continued pretraining. First 500 characters of the text:")
print(dataset[0]["text"][:500])

For continued pretraining with pure text, we **do not** use conversational data standardization or apply a formatting function that transforms conversational data. The `dataset` object directly contains your pure text in the `text` field, ready for the trainer.

In [ ]:
# New Trainer Setup Cell for Continued Pretraining

from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset, # Use the dataset created from your pure text
    dataset_text_field = "text", # This specifies that the 'text' field contains the training data
    max_seq_length = max_seq_length,
    packing = True, # Highly recommended for continued pretraining for efficiency
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 500, # Increased max_steps to give the model more time to learn
        learning_rate = 2e-5,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

For continued pretraining on pure text, we **do NOT** use `train_on_responses_only`. This function is designed for masking parts of *conversational* data, and in pure text pretraining, the model learns from the entire text provided in the `text` field.

In [ ]:
# Verification: Raw text content from the first dataset entry.
# The SFTTrainer with packing=True will handle tokenization and label creation internally.

print("Raw text content from the first dataset entry (what the trainer sees):\n")
print(dataset[0]["text"][:1000]) # Display first 1000 characters for inspection

In [ ]:
# @title Show current memory stats before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved before training.")

In [ ]:
print("Starting model training...")
trainer_stats = trainer.train()
print("Training completed.")

In [ ]:
# @title Show final memory and time stats after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

Masked labels inspection (as typically done for conversational finetuning) is **not applicable** for pure text continued pretraining. The trainer handles tokenization and label creation internally for language modeling, where the model learns to predict the next token based on the preceding ones throughout the entire text.

For continued pretraining with pure text, we **do not** use conversational data standardization or apply a formatting function that transforms conversational data. The `dataset` object directly contains your pure text in the `text` field, ready for the trainer.

In [ ]:
FastLanguageModel.for_inference(model) # Ensure inference mode is enabled

messages = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'.\n\n**Ingredients:**\n* 1 drop of morning dew from a forgotten forest\n* 3 petals from a moonflower\n* 1 pinch of stardust\n\n**Preparation Steps:**\n1. Collect dew at dawn.\n2. Gently press moonflower petals into a vial.\n3. Add stardust to activate.\n\nNow, create a magical recipe for an 'Elixir of Dreams'. List the mystical ingredients needed and the detailed preparation steps to brew it."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 0.9, min_p = 0.6) # Slightly adjusted parameters

In [ ]:
FastLanguageModel.for_inference(model) # Ensure inference mode is enabled

messages = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'.\n\n**Ingredients:**\n* 1 drop of morning dew from a forgotten forest\n* 3 petals from a moonflower\n* 1 pinch of stardust\n\n**Preparation Steps:**\n1. Collect dew at dawn.\n2. Gently press moonflower petals into a vial.\n3. Add stardust to activate.\n\nNow, create a magical recipe for an 'Elixir of Dreams'. List the mystical ingredients needed and the detailed preparation steps to brew it."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 1.5, min_p = 0.1) # Slightly adjusted parameters

In [ ]:
FastLanguageModel.for_inference(model) # Ensure inference mode is enabled

messages = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'.\n\n**Ingredients:**\n* 1 drop of morning dew from a forgotten forest\n* 3 petals from a moonflower\n* 1 pinch of stardust\n\n**Preparation Steps:**\n1. Collect dew at dawn.\n2. Gently press moonflower petals into a vial.\n3. Add stardust to activate.\n\nNow, create a magical recipe for an 'Elixir of Dreams'. List the mystical ingredients needed and the detailed preparation steps to brew it."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 1.5, min_p = 0.1) # Slightly adjusted parameters

In [ ]:
FastLanguageModel.for_inference(model) # Ensure inference mode is enabled

messages = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Dreams'. List the mystical ingredients needed and the detailed preparation steps to brew it."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

In [ ]:
# --- Compare with the base model to see the effect of pretraining ---
# Uncomment the following block to load the base model for comparison

if True:
    from unsloth import FastLanguageModel
    # Load the base model without any LoRA adapters
    base_model, base_tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Llama-3.2-3B-bnb-4bit", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    base_tokenizer = get_chat_template(
        base_tokenizer,
        chat_template = "llama-3.1",
    )
    FastLanguageModel.for_inference(base_model) # Enable native 2x faster inference
else:
    base_model = model
    base_tokenizer = tokenizer

print("Model loaded for comparison (either base or finetuned depending on uncommenting)...")

In [ ]:
# Run the same inference prompt on the (potentially) base model
messages_for_comparison = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'.\n\n**Ingredients:**\n* 1 drop of morning dew from a forgotten forest\n* 3 petals from a moonflower\n* 1 pinch of stardust\n\n**Preparation Steps:**\n1. Collect dew at dawn.\n2. Gently press moonflower petals into a vial.\n3. Add stardust to activate.\n\nNow, create a magical recipe for an 'Elixir of Dreams'. List the mystical ingredients needed and the detailed preparation steps to brew it."},
]
inputs_for_comparison = base_tokenizer.apply_chat_template(
    messages_for_comparison,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer_comparison = TextStreamer(base_tokenizer, skip_prompt = True)
print("\n--- Output from Comparison Model ---")
_ = base_model.generate(input_ids = inputs_for_comparison, streamer = text_streamer_comparison, max_new_tokens = 256,
                   use_cache = True, temperature = 1.5, min_p = 0.1)
print("\n--- End of Comparison Output ---")

print("\nNow, compare this output to the previous one from your finetuned model.")

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
# New Cell for Model Loading (from original Znh96WvcstqF)

from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit", # Changed to base model
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

In [ ]:
# New Cell for LoRA Adapters (from original 6bZsfBuZDeCL)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
# New Trainer Setup Cell for Continued Pretraining

from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset, # Use the dataset created from your pure text
    dataset_text_field = "text", # This specifies that the 'text' field contains the training data
    max_seq_length = max_seq_length,
    packing = True, # Highly recommended for continued pretraining for efficiency
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100, # Increased max_steps to give the model more time to learn
        learning_rate = 2e-5,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
print("Starting model training...")
trainer_stats = trainer.train()
print("Training completed.")

In [ ]:
FastLanguageModel.for_inference(model) # Ensure inference mode is enabled

messages = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 1.5, min_p = 0.1) # Slightly adjusted parameters

In [ ]:
FastLanguageModel.for_inference(model) # Ensure inference mode is enabled

messages = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 0.9, min_p = 0.6) # Slightly adjusted parameters

# Without training

In [ ]:
# --- Compare with the base model to see the effect of pretraining ---
# Uncomment the following block to load the base model for comparison

if True:
    from unsloth import FastLanguageModel
    # Load the base model without any LoRA adapters
    base_model, base_tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    base_tokenizer = get_chat_template(
        base_tokenizer,
        chat_template = "llama-3.1",
    )
    FastLanguageModel.for_inference(base_model) # Enable native 2x faster inference
else:
    base_model = model
    base_tokenizer = tokenizer

print("Model loaded for comparison (either base or finetuned depending on uncommenting)...")

In [ ]:
# Run the same inference prompt on the (potentially) base model
messages_for_comparison = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'."},
]
inputs_for_comparison = base_tokenizer.apply_chat_template(
    messages_for_comparison,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer_comparison = TextStreamer(base_tokenizer, skip_prompt = True)
print("\n--- Output from Comparison Model ---")
_ = base_model.generate(input_ids = inputs_for_comparison, streamer = text_streamer_comparison, max_new_tokens = 256,
                   use_cache = True, temperature = 1.5, min_p = 0.1)
print("\n--- End of Comparison Output ---")

print("\nNow, compare this output to the previous one from your finetuned model.")

In [ ]:
# New Trainer Setup Cell for Continued Pretraining

from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset, # Use the dataset created from your pure text
    dataset_text_field = "text", # This specifies that the 'text' field contains the training data
    max_seq_length = max_seq_length,
    packing = True, # Highly recommended for continued pretraining for efficiency
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 300, # Increased max_steps to give the model more time to learn
        learning_rate = 2e-5,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
print("Starting model training...")
trainer_stats = trainer.train()
print("Training completed.")

In [ ]:
FastLanguageModel.for_inference(model) # Ensure inference mode is enabled

messages = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 1.5, min_p = 0.1) # Slightly adjusted parameters

In [ ]:
FastLanguageModel.for_inference(model) # Ensure inference mode is enabled

messages = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 0.9, min_p = 0.6) # Slightly adjusted parameters